# 601. Human Traffic of Stadium

## Problem Description
X city built a new stadium, and each day many people visit it. The stats are saved in the **Stadium** table with columns: `id`, `visit_date`, and `people`.

We need to:
- Display the records where there are **3 or more consecutive rows** with `people >= 100`.  
- Consecutive means continuous days in the dataset.  
- Output should include all rows that are part of such consecutive sequences.

---

## Schema

### Table: Stadium
| Column Name | Type    | Description                          |
|-------------|---------|--------------------------------------|
| id          | INT     | Unique identifier for each record     |
| visit_date  | DATE    | Date of the visit                    |
| people      | INT     | Number of people visiting that day    |

**Primary Key:** `id`

---

## Sample Data

### Input: Stadium
| id | visit_date | people |
|----|------------|--------|
| 1  | 2017-01-01 | 10     |
| 2  | 2017-01-02 | 109    |
| 3  | 2017-01-03 | 150    |
| 4  | 2017-01-04 | 99     |
| 5  | 2017-01-05 | 145    |
| 6  | 2017-01-06 | 1455   |
| 7  | 2017-01-07 | 199    |
| 8  | 2017-01-08 | 188    |

---

## Expected Output

| id | visit_date | people |
|----|------------|--------|
| 5  | 2017-01-05 | 145    |
| 6  | 2017-01-06 | 1455   |
| 7  | 2017-01-07 | 199    |
| 8  | 2017-01-08 | 188    |

---

## Explanation
- Rows 2 and 3 have people ≥ 100, but only 2 consecutive days → not enough.  
- Rows 5, 6, 7, 8 form a sequence of 4 consecutive days with people ≥ 100 → included in output.  

---

## PySpark Code: Create DataFrame and Temp View

```python


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, DateType

# Schema for Stadium
stadium_schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("visit_date", DateType(), False),
    StructField("people", IntegerType(), False)
])

# Data for Stadium
from datetime import date
stadium_data = [
    (1, date(2017,1,1), 10),
    (2, date(2017,1,2), 109),
    (3, date(2017,1,3), 150),
    (4, date(2017,1,4), 99),
    (5, date(2017,1,5), 145),
    (6, date(2017,1,6), 1455),
    (7, date(2017,1,7), 199),
    (8, date(2017,1,8), 188)
]

# Create DataFrame
stadium_df = spark.createDataFrame(stadium_data, stadium_schema)

# Register Temp View
stadium_df.createOrReplaceTempView("Stadium")

# Quick check
stadium_df.show()


In [0]:
%sql
WITH cte AS (
		SELECT id - row_number() OVER (
				ORDER BY id ASC
				) AS diff,
			*
		FROM stadium
		WHERE people >= 100
		),
	cte2(SELECT count(diff) OVER (PARTITION BY diff) AS cnt, * FROM cte)

SELECT id,
	visit_date,
	people
FROM cte2
WHERE cnt >= 3
